# Do daily visual checks of data quality
## L. Caffarello, July 2022
### Updates by A. Ordog, August-September 2022
### August 25, 2022 (AO): 
#### - changed filenames to account for survey phases
#### - changed frequency channels read in to all channels (instead of every 12th)
#### - included 5 MHz to each side of the central frequency in the single-scan, 1D timeseries plots
#### - included persistent RFI mask indication (grey background) on timeseries plots
### September 12, 2022 (AO):
#### - option to mask out persistent RFI
#### - indicate elevation of scans
#### - scan IDs indicated in waterfall plot columns
### January 5, 2023 (AO):
#### - option to plot scan in 2D in Galactic coordinates
### January 11, 2023 (AO):
#### - fixed formatting for reading in phase and day (only need to select it in first cell)
#### - included azimuth axis on 1D single-scan time-domain plots
#### - added option to change frequency (default 520 MHz) and colorscale in 2D Galactic plot
### July 4, 2024 (AO):
#### This copy for P. Hembling to look for Meridian satellites

## Import packages and select phase and day of survey:

In [1]:
#import imp
import os
import subprocess
import h5py
import numpy as np
from astropy.time import Time
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
import datetime
import matplotlib.dates as mdates
from matplotlib.dates import HourLocator as HourLocator
from matplotlib.dates import MinuteLocator as MinuteLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable
from astropy import units as u
from astropy.time import TimeDelta
from ipywidgets import interact
from ipywidgets import fixed
from ipywidgets import interactive_output
from astropy.coordinates import SkyCoord,EarthLocation, AltAz, ICRS, Galactic, FK4, FK5

#### Change the directory to where the files are located ####
day = 7
phase = 3
directory = '/srv/data/dva/survey_azimuth_scans/' # This is where the raw scans are stored
##############################################################

## Read in the file listing azimuth scan start and stop times:
### The scans were NOT done sequentially by their scan IDs; there is some randomness to the order.
### In order to read in the scans that were done on a particular day, we use the files that have this format:
#### DVAsurvey_phase3_day007.txt

In [2]:
scan_id = []    # The scan id number
scan_start = []  # Start time of the scan (UTC)
scan_stop = []   # Stop time of the scan (UTC)
scan_el = []

# Read in the data and store it in arrays:
with open(directory+'DVAsurvey_phase'+str(phase)+'_day'+f"{day:03}"+'.txt') as fp:
    for line in fp:       
        scan_id.append(int(line.split()[0]))
        scan_start.append(line.split()[1]+'T'+line.split()[2][0:12])
        scan_stop.append(line.split()[3]+'T'+line.split()[4][0:12])
        scan_el.append(line.split()[5][0:2])
        
# Print out the scan numbers with their start and stop times:
for i in range(0,len(scan_id)):
    print(f"{scan_id[i]:04}",scan_start[i],scan_stop[i],scan_el[i])

# Convert start and stop times to Modified Julian Day (MJD).
# This is needed for plotting and for selecting out data collected
# between particular times:
scan_start_mjd = Time(scan_start, format='isot',scale='utc').mjd
scan_stop_mjd  = Time(scan_stop,  format='isot',scale='utc').mjd


0929 2022-12-04T00:50:58.727 2022-12-04T01:09:00.963 49
0930 2022-12-04T01:11:19.215 2022-12-04T01:29:21.356 49
2443 2022-12-04T01:34:03.562 2022-12-04T01:52:05.390 20
1148 2022-12-04T01:56:47.655 2022-12-04T02:14:49.934 49
2661 2022-12-04T02:19:32.111 2022-12-04T02:37:34.297 20
2662 2022-12-04T02:39:52.975 2022-12-04T02:57:54.850 19
2663 2022-12-04T02:59:37.353 2022-12-04T03:17:39.334 20
2664 2022-12-04T03:19:22.400 2022-12-04T03:37:24.577 20
2593 2022-12-04T03:39:43.111 2022-12-04T03:57:45.099 20
1298 2022-12-04T04:02:27.013 2022-12-04T04:20:28.817 49
1299 2022-12-04T04:22:48.022 2022-12-04T04:40:50.109 49
1300 2022-12-04T04:42:32.835 2022-12-04T05:00:34.581 49
1301 2022-12-04T05:02:17.195 2022-12-04T05:20:19.424 49
1302 2022-12-04T05:22:38.263 2022-12-04T05:40:40.507 49
1303 2022-12-04T05:42:23.048 2022-12-04T06:00:25.100 49
2816 2022-12-04T06:05:06.968 2022-12-04T06:23:09.124 20
0083 2022-12-04T06:28:27.166 2022-12-04T06:46:29.198 49
1596 2022-12-04T06:51:11.274 2022-12-04T07:09:13

## Read in scan files and stitch them together:
### Note: this may take several minutes, depending on the number of scans done that day

In [ ]:
%%time
t_set = []
az_set = []
dec_set = []
ra_set = []
el_set = []
noise_set = []
trim_flag = []

scan0 = f"{scan_id[0]:04}"

# Use one of the scans to get the list of frequencies:
file = h5py.File(directory+'dva_survey_phase'+str(phase)+'_raw_'+scan0+'.h5','r')
freq = file['data']['beam_0']['band_SB0']['frequency'][:]/1e6

# Create empty arrays for the power data:
RR_set = np.empty([0,len(freq)])
LL_set = np.empty([0,len(freq)])
reRL_set = np.empty([0,len(freq)])
imRL_set = np.empty([0,len(freq)])

# Loop through all the scans in the "scan_num" list:
for i in scan_id:
#for i in scan_id[0:5]:
    print(i)
    # select the file:
    file = h5py.File(directory+'dva_survey_phase'+str(phase)+'_raw_'+f"{i:04}"+'.h5','r')
    print(file)
    
    # access the correct location in the file structure:
    dataset = file['data']['beam_0']['band_SB0']['scan_0']
    
    # Add the position and time data to the corresponding arrays:
    dec_set = np.concatenate([dec_set,dataset['metadata']['declination']])
    ra_set = np.concatenate([ra_set,dataset['metadata']['right_ascension']])
    el_set = np.concatenate([el_set,dataset['metadata']['elevation']])
    az_set = np.concatenate([az_set,dataset['metadata']['azimuth']])
    t_set = np.concatenate([t_set,dataset['metadata']['utc']])
    noise_set = np.concatenate([noise_set,dataset['metadata']['noise_state']]) #This is a "mask" for noise regions 1 = noise 0=all good
    trim_flag = np.concatenate([trim_flag,dataset['metadata']['trim_scan_flag']])
    
    # Add the spectrometer power data to the corresponding arrays:
    RR_set = np.concatenate([RR_set,dataset['data'][:,0,:]],axis=0)
    LL_set = np.concatenate([LL_set,dataset['data'][:,1,:]],axis=0)
    reRL_set = np.concatenate([reRL_set,dataset['data'][:,2,:]],axis=0)
    imRL_set = np.concatenate([imRL_set,dataset['data'][:,3,:]],axis=0)
    
t_plt = Time(t_set, format='isot',scale='utc').mjd

## Read in persistent RFI mask

In [ ]:
i = 0
RFI_mask_idx = []
with open('/srv/data/dva/RFIpersist_mask/RFIpersist_mask.txt') as fp:
#with open('../DVA2/DATA/PersistRFI_v01_Jul13.txt') as fp:
    for line in fp:
        if i>0: 
            #print(line)
            RFI_mask_idx.append(int(line.split()[0]))
        i=i+1

## Updated visualization, including single-scan option
### Changes made by AO:
#### - power bounds in dB to see lower power signal
#### - changed to log scale and added RR to time series plots and spectra
#### - added grids to 1D plots
#### - added option for single-scan plots (both waterfall and 1D)

In [ ]:
df = freq[1]-freq[0]
power_min = 65
power_max = 79
fs = 14

def DVA_Waterfall_View(RFI_masked):  
    
    #TODO: implement toggling between RR, LL,reRL, imRL
       
    fig,axs = plt.subplots(1,1,figsize=(15,10)) 
    
    for i in range(0,len(scan_id)):
        w = np.where((t_plt>=scan_start_mjd[i]) & (t_plt<=scan_stop_mjd[i]))[0]
        extent = [scan_start_mjd[i],scan_stop_mjd[i],freq[0],freq[-1]]
        
        if scan_el[i] == '49':
            textclr = 'blue'
        else:
            textclr = 'red'
                
        data_plot = 10*np.log10(LL_set[w,:].T)
        if RFI_masked:
            data_plot[RFI_mask_idx,:] = np.nan
    
        im = axs.imshow(data_plot,aspect='auto',vmin=power_min,vmax=power_max,
                        origin='lower',extent=extent,cmap='viridis')

        axs.text(scan_start_mjd[i]+3e-3,1038,f"{scan_id[i]:04}",rotation=45,fontsize=fs,color=textclr)
    
    divider = make_axes_locatable(axs)
    cax = divider.append_axes('right', size='2%', pad=0.05)
    cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    cbar.ax.tick_params(labelsize=fs) 
    cbar.set_label('Power (dB)', fontsize=fs)

    axs.set_xlim(t_plt[0],t_plt[-1])
    axs.set_ylim(freq[0],freq[-1])
    axs.tick_params(axis='both', labelsize=fs)
    axs.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axs.fmt_xdata = mdates.DateFormatter('%H:%M:%S')
    axs.set_xlabel('Time (UTC)',fontsize=fs)        
    axs.set_ylabel('Frequency (MHz)',fontsize=fs)
    
    return

def DVA_Waterfall_View_singlescan(scan_chosen, RFI_masked):
    
    #print(axmap)
    #TODO: implement toggling between RR, LL,reRL, imRL
    scan_id_plot = scan_chosen
    scan_idx = np.where(np.array(scan_id) == scan_id_plot)    
    w = np.where((t_plt>=scan_start_mjd[scan_idx]) & (t_plt<=scan_stop_mjd[scan_idx]))[0]
        
    fig,axs = plt.subplots(1,1,figsize=(15,10))     
    extent = [scan_start_mjd[scan_idx][0],scan_stop_mjd[scan_idx][0],freq[0],freq[-1]]
    
    data_plot = 10*np.log10(LL_set[w,:].T)
    if RFI_masked:
        data_plot[RFI_mask_idx,:] = np.nan
    
    im = axs.imshow(data_plot,aspect='auto',vmin=power_min,vmax=power_max,
                    origin='lower',extent=extent,cmap='viridis')

    divider = make_axes_locatable(axs)
    cax = divider.append_axes('right', size='2%', pad=0.05)
    cbar = fig.colorbar(im, cax=cax, orientation='vertical')
    cbar.ax.tick_params(labelsize=fs) 
    cbar.set_label('Power (dB)', fontsize=fs)

    axs.set_xlim(extent[0],extent[1])
    axs.set_ylim(freq[0],freq[-1])
    axs.tick_params(axis='both', labelsize=fs)
    axs.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axs.fmt_xdata = mdates.DateFormatter('%H:%M:%S')
    axs.set_xlabel('Time (UTC)',fontsize=fs)        
    axs.set_ylabel('Frequency (MHz)',fontsize=fs)
    axs.set_title('Scan '+str(scan_id_plot)+' , el = '+str(scan_el[scan_idx[0][0]]),fontsize=fs)
    
    return


def DVA_Cross_Sections(freq_chosen, time_idx, freq_cross_section,RFI_masked):
    
    freq_measured = np.where(abs(freq-freq_chosen)<df)[0][0]
    freq_below = np.where((freq>freq_chosen-5) & (freq<freq_chosen))[0]
    freq_above = np.where((freq<freq_chosen+5) & (freq>freq_chosen))[0]
   
    fig,axs1 = plt.subplots(1,1,figsize=(16,6))  

    if(freq_cross_section):
        
        data_plot_L = 10*np.log10(LL_set[time_idx,:])
        data_plot_R = 10*np.log10(RR_set[time_idx,:])
        
        if RFI_masked:
            data_plot_L[RFI_mask_idx] = np.nan
            data_plot_R[RFI_mask_idx] = np.nan

        axs1.plot(freq,data_plot_L,label='LL',color='blue')
        axs1.plot(freq,data_plot_R,label='RR',color='red')        
        axs1.vlines(freq_chosen, 0 , 100e9, color = 'purple')
        axs1.set_ylim(power_min,power_max) 
        axs1.set_xlim(350,1050)
        axs1.set_xlabel('Frequency',fontsize=fs)
        axs1.tick_params(axis='both',labelsize=fs)
    else:
        axs1.scatter(t_plt, 10*np.log10(LL_set[:,freq_measured]),label='LL',s=0.8,zorder=1,color='blue')
        axs1.scatter(t_plt, 10*np.log10(RR_set[:,freq_measured]),label='RR',s=0.8,zorder=1,color='red')
          
        axs1.vlines(t_plt[time_idx], 0 , 100e9, color = 'purple')
        
        for i in range(0,len(scan_id)):
            if scan_el[i] == '49':
                elclr = 'C0'
            else:
                elclr = 'C1'        
            axs1.axvspan(scan_start_mjd[i],scan_stop_mjd[i],color=elclr,alpha=0.2,zorder=0)
            axs1.text(scan_start_mjd[i]+3e-3,power_max+0.5,f"{scan_id[i]:04}",rotation=45,fontsize=fs)

        axs1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        axs1.set_ylim(power_min,power_max)
        axs1.fmt_xdata = mdates.DateFormatter('%H:%M:%S')
        axs1.set_xlabel('Time (UTC)',fontsize=fs)
        axs1.set_xlim(t_plt[0],t_plt[-1])
        axs1.tick_params(axis='both',labelsize=fs)
        axs1.tick_params(axis='both',labelsize=fs)
        
        if freq_measured in RFI_mask_idx:
            axs1.set_facecolor('lightgray')        
        
    axs1.set_ylabel('Power',fontsize=fs)
    axs1.legend(fontsize=fs,markerscale=5)
    axs1.grid()
    
    return

    
def DVA_Cross_Sections_singlescan(freq_chosen,scan_chosen,RFI_masked,freq_cross_section,time_idx):
    
    freq_measured = np.where(abs(freq-freq_chosen)<df)[0][0]
    freq_below = np.where((freq>freq_chosen-5) & (freq<freq_chosen))[0]
    freq_above = np.where((freq<freq_chosen+5) & (freq>freq_chosen))[0]
          
    scan_id_plot = scan_chosen
    scan_idx = np.where(np.array(scan_id) == scan_id_plot)
    
    w = np.where((t_plt>=scan_start_mjd[scan_idx]) & (t_plt<=scan_stop_mjd[scan_idx]))[0]
         
    fig,axs1 = plt.subplots(1,1,figsize=(16,6)) 
            
    if(freq_cross_section):
            
        data_plot_L = 10*np.log10(LL_set[w[time_idx],:])
        data_plot_R = 10*np.log10(RR_set[w[time_idx],:])
        
        if RFI_masked:
            data_plot_L[RFI_mask_idx] = np.nan
            data_plot_R[RFI_mask_idx] = np.nan
            
        axs1.plot(freq,data_plot_L, label='LL',color='blue')
        axs1.plot(freq,data_plot_R, label='RR',color='red')
        axs1.vlines(freq_chosen, 0 , 100e9, color = 'purple')
        axs1.set_ylim(power_min,power_max)
        axs1.set_xlim(350,1030)
        axs1.set_xlabel('Frequency',fontsize=fs)
        axs1.tick_params(axis='both',labelsize=fs)
    else:
        axs1.plot(t_plt[w], 10*np.log10(LL_set[w,freq_measured]), label='LL',color='blue',zorder=1)
        axs1.plot(t_plt[w], 10*np.log10(RR_set[w,freq_measured]), label='RR',color='red',zorder=1)
        for ifreq in freq_above:
            axs1.plot(t_plt[w], 10*np.log10(LL_set[w,ifreq]), alpha=0.5,color='C0',zorder=0)
            axs1.plot(t_plt[w], 10*np.log10(RR_set[w,ifreq]), alpha=0.5,color='C1',zorder=0)
        for ifreq in freq_below:
            axs1.plot(t_plt[w], 10*np.log10(LL_set[w,ifreq]), alpha=0.5,color='C9',zorder=0)
            axs1.plot(t_plt[w], 10*np.log10(RR_set[w,ifreq]), alpha=0.5,color='salmon',zorder=0)
        axs1.vlines(t_plt[w][time_idx], 0 , 100e9, color = 'purple')
        axaz = axs1.twiny()
        axaz.set_xlim(az_set[w][0],az_set[w][-1])
        axaz.tick_params(axis='x',labelsize=fs)

        axs1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        axs1.set_ylim(power_min,power_max) #AO changed this to log scale limits
        axs1.fmt_xdata = mdates.DateFormatter('%H:%M:%S')
        axs1.set_xlabel('Time (UTC)',fontsize=fs)
        axs1.set_xlim(t_plt[w][0],t_plt[w][-1])
        axs1.tick_params(axis='both',labelsize=fs)
        axs1.set_title('Scan '+str(scan_id_plot)+' , el = '+str(scan_el[scan_idx[0][0]]),fontsize=fs)
            
        if freq_measured in RFI_mask_idx:
             axs1.set_facecolor('lightgray')
        
    axs1.set_ylabel('Power',fontsize=fs)
    axs1.legend(fontsize=fs,markerscale=5)
    axs1.grid()
    
    return
    
def make_map(scan_pick,freq_pick=520.,dB1=74.,dB2=78.,*args,**kwargs):

    fs=12
    spts = 5
    frq_idx = np.where(abs(freq-freq_pick)<df)[0][0]
    
    scan_idx = np.where(np.array(scan_id) == scan_pick)[0][0]
    print('scan: ', scan_id[scan_idx])
    print('frequency: ',freq_pick)
    print('power range (dB): ',dB1,dB2)
    
    start_plt = Time(scan_start[scan_idx], format='isot',scale='utc').mjd
    stop_plt = Time(scan_stop[scan_idx], format='isot',scale='utc').mjd        
    t_idx = np.where((t_plt>=start_plt) & (t_plt<=stop_plt))[0]
        
    loc = EarthLocation(lat = 49.32*u.deg, lon = -119.62*u.deg, height = 546*u.m)
    AltAzcoord = SkyCoord(alt=el_set[t_idx]*u.deg, az=az_set[t_idx]*u.deg, 
                      obstime=Time(t_set[t_idx],format='isot',scale='utc',location=loc),
                      frame = 'altaz', location = loc)
    l_scan = AltAzcoord.galactic.l.deg
    b_scan = AltAzcoord.galactic.b.deg    
    gal = SkyCoord(l_scan, b_scan, frame='galactic', unit=u.deg)
    
    wnoise = np.where(noise_set[t_idx] == 1)[0]
    data_plot = 10*np.log10(LL_set[t_idx,frq_idx])
    data_plot[wnoise] = np.nan
    
    fig = plt.figure(figsize=(14,7))
    plt.subplot(111, projection='aitoff')
    plt.scatter(-gal.l.wrap_at('180d').radian, gal.b.radian,s=spts,c=data_plot,
                vmin=dB1,vmax=dB2,alpha=1)
    positions = (-150*np.pi/180.,-120*np.pi/180.,-90*np.pi/180.,-60*np.pi/180.,-30*np.pi/180.,0,
                    30*np.pi/180.,60*np.pi/180.,90*np.pi/180.,120*np.pi/180.,150*np.pi/180.)
    labels = ("150","120","90","60","30","0","330","300","270","240","210")
    plt.xticks(positions, labels,fontsize=fs)
    plt.yticks(fontsize=fs)
    plt.xlabel('Galactic Longitude (degrees)',fontsize=fs)
    plt.ylabel('Galactic Latitude (degrees)',fontsize=fs)
    plt.grid(True)

    return 

def make_map_eq(scan_pick,freq_pick=520.,dB1=74.,dB2=78.,*args,**kwargs):

    fs=12
    spts = 5
    frq_idx = np.where(abs(freq-freq_pick)<df)[0][0]
    
    scan_idx = np.where(np.array(scan_id) == scan_pick)[0][0]
    print('scan: ', scan_id[scan_idx])
    print('frequency: ',freq_pick)
    print('power range (dB): ',dB1,dB2)
    
    start_plt = Time(scan_start[scan_idx], format='isot',scale='utc').mjd
    stop_plt = Time(scan_stop[scan_idx], format='isot',scale='utc').mjd        
    t_idx = np.where((t_plt>=start_plt) & (t_plt<=stop_plt))[0]
        
    loc = EarthLocation(lat = 49.32*u.deg, lon = -119.62*u.deg, height = 546*u.m)
    AltAzcoord = SkyCoord(alt=el_set[t_idx]*u.deg, az=az_set[t_idx]*u.deg, 
                      obstime=Time(t_set[t_idx],format='isot',scale='utc',location=loc),
                      frame = 'altaz', location = loc)
    l_scan = AltAzcoord.icrs.ra.deg
    b_scan = AltAzcoord.icrs.dec.deg   
    gal = SkyCoord(l_scan, b_scan, frame='icrs', unit=u.deg)
    
    wnoise = np.where(noise_set[t_idx] == 1)[0]
    data_plot = 10*np.log10(LL_set[t_idx,frq_idx])
    data_plot[wnoise] = np.nan
    
    fig = plt.figure(figsize=(14,7))
    #plt.subplot(111, projection='cartesian')
    plt.subplot(111)
    plt.scatter(-gal.ra.wrap_at('180d').radian, gal.dec.deg,s=spts,c=data_plot,
                vmin=dB1,vmax=dB2,alpha=1)
    positions = (-150*np.pi/180.,-120*np.pi/180.,-90*np.pi/180.,-60*np.pi/180.,-30*np.pi/180.,0,
                    30*np.pi/180.,60*np.pi/180.,90*np.pi/180.,120*np.pi/180.,150*np.pi/180.)
    labels = ("150","120","90","60","30","0","330","300","270","240","210")
    plt.xticks(positions, labels,fontsize=fs)
    plt.yticks(fontsize=fs)
    #plt.xlabel('Galactic Longitude (degrees)',fontsize=fs)
    #plt.ylabel('Galactic Latitude (degrees)',fontsize=fs)
    plt.grid(True)

    return 
    
        
def DVA_Visualization(waterfall_enabled,singlescan):

    if waterfall_enabled:
        if singlescan:
            interact(DVA_Waterfall_View_singlescan,scan_chosen = scan_id, RFI_masked = False)
        else:
            interact(DVA_Waterfall_View, RFI_masked = False)
    else:
        if singlescan:
            interact(DVA_Cross_Sections_singlescan, freq_chosen = (350, 1030, df), 
                     scan_chosen = scan_id, RFI_masked = False, freq_cross_section = False,
                    time_idx = (0,1802))
        else:
            interact(DVA_Cross_Sections, freq_chosen = (350, 1030, df), 
                     freq_cross_section = False, time_idx = (0,len(t_plt)-1), RFI_masked = False)
            
    return

In [ ]:
interact(DVA_Visualization,waterfall_enabled = True, singlescan = False)

In [ ]:
make_map_eq(1309,freq_pick=860,dB1=66,dB2=76)

In [ ]:
make_map(1115,freq_pick=860,dB1=69,dB2=71)